# 04. 가설 검증과 판단

분석 시점은 2020년 1월 초다. 2019년 한 해의 거래 이력으로 고객 1,468명의 접촉 대상을 좁힌다. 먼저 전체 기준선을 잡고(0장), 2019년 구매 이력으로 만든 RFM 가치 등급으로 나눠 누가 가치 있는지(1장), 등급마다 언제 돌아오는지(2장), 같은 등급 안에서도 구매 상태가 갈리는지(3장)를 차례로 본다. 각 절에서 대상과 분모를 먼저 정하고, 관찰 결과와 판단을 구분한다.


In [1]:
import json
import io
import contextlib
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p/'notebooks'/'02_data_preparation.ipynb').exists())
prep_path = root/'notebooks'/'02_data_preparation.ipynb'
prep = json.loads(prep_path.read_text(encoding='utf-8'))
for cell in prep['cells']:
    if cell['cell_type']=='code' and 'pipeline' in cell.get('metadata',{}).get('tags',[]):
        with contextlib.redirect_stdout(io.StringIO()):
            exec(compile(''.join(cell['source']),str(prep_path),'exec'),globals())
grade_colors={'Bronze':'#CD7F32','Silver':'#C0C0C0','Gold':'#FFD700',
              'Platinum':'#E0B0FF','Diamond':'#B9F2FF'}
print('분석 테이블 준비 완료')


분석 테이블 준비 완료


## 분석 대상과 공통 기준

2019년 상품라인 52,924행을 고객ID와 거래ID 기준 고객 주문 26,631건, 고객 1,468명으로 집계한다. 연간 재구매 고객은 서로 다른 날 2일 이상 구매한 고객이며, 상품매출은 배송비를 제외한 세후 금액이다. 각 장의 분모와 집계 단위는 해당 절에서 따로 밝힌다. 등급과 세그먼트는 2019-12-31을 기준일로 한 구매 이력 프로파일이다.


## 0. 분석 시점과 전체 기준선

<strong>분석 기준:</strong> 고객 전체의 <strong>연간 재구매 고객 비율</strong>, 재구매 여부로 나눈 두 집단의 <strong>상품매출 비중</strong>, <strong>처음 구매한 달 이후의 경과월별 재구매율</strong>을 차례로 본다. 연간 재구매 고객 비율은 서로 다른 구매일이 2일 이상인 고객 / 1,468명이다. 경과월별 재구매율의 기준월은 2019년 거래에서 첫 구매가 확인된 달이며, 각 경과월을 관찰할 수 있는 고객을 분모로 한 고객 가중 코호트 재구매율이다.

### 0-1. 서로 다른 날 다시 구매한 고객은 얼마나 되는가?


In [2]:
retention = pd.DataFrame([
    ['연간 재구매 고객 비율', int(analysis_customers['다른날재방문'].sum()), len(analysis_customers)],
    ['첫 구매 다음 달 재구매율',
     int(analysis_customers.loc[analysis_customers['다음달관찰가능'],'첫구매월+1개월복귀'].sum()),
     int(analysis_customers['다음달관찰가능'].sum())]
],columns=['지표','복귀 고객','대상 고객'])
retention['비율(%)'] = (retention['복귀 고객']/retention['대상 고객']*100).round(2)
display(retention.rename(columns={'복귀 고객':'재구매 고객'}))


,지표,재구매 고객,대상 고객,비율(%)
0,연간 재구매 고객 비율,734,1468,50.00
1,첫 구매 다음 달 재구매율,130,1362,9.54


In [3]:
total_n=len(analysis_customers)
visit_dist=(pd.cut(analysis_customers['구매일수'],
    bins=[0,1,2,5,10,float('inf')],
    labels=['1일(다른 날 재구매 없음)','2일','3-5일','6-10일','11일 이상'])
    .value_counts().reindex(
        ['1일(다른 날 재구매 없음)','2일','3-5일','6-10일','11일 이상']
    ).reset_index())
visit_dist.columns=['구매일수 구간','고객수']
visit_dist['비중']=(visit_dist['고객수']/total_n*100).round(1)
assert int(visit_dist['고객수'].sum())==1468
display(visit_dist)


,구매일수 구간,고객수,비중
0,1일(다른 날 재구매 없음),734,50.0
1,2일,364,24.8
2,3-5일,291,19.8
3,6-10일,68,4.6
4,11일 이상,11,0.7


In [4]:
fig=go.Figure(go.Bar(
    x=visit_dist['구매일수 구간'],y=visit_dist['고객수'],
    text=[f'{c:,}명<br>({r}%)' for c,r in zip(visit_dist['고객수'],visit_dist['비중'])],
    textposition='outside',marker_color=['#95a5a6']+['#3498db']*4,
    cliponaxis=False,showlegend=False))
fig.update_layout(title=dict(
    text='고객별 연간 구매일 수 분포<br><sub>전체 1,468명 · 구매일 2일 이상 734명 (50.0%)</sub>',
    x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    xaxis_title='서로 다른 구매일 수',yaxis_title='고객 수',
    plot_bgcolor='white',height=440,margin=dict(t=110))
fig.update_xaxes(gridcolor='#ececec')
fig.update_yaxes(gridcolor='#ececec',tickformat=',',
    range=[0,visit_dist['고객수'].max()*1.15])
fig


구매일이 하루뿐인 고객은 734명이다. 다른 날에도 구매한 고객 역시 734명(50.0%)이다. 이 값은 2019년 안에 다른 날 구매가 한 번이라도 있었는지를 센다. 돌아온 시점은 0-3에서 본다.


### 0-2. 재구매 고객이 상품매출에서 차지하는 비중은?

- 집단 1: 서로 다른 날 2일 이상 구매한 고객 734명
- 집단 2: 구매일이 하루뿐인 고객 734명
- 기준: 고객별 누적 상품매출(배송비 제외)의 합과 전체 $4,840,102.19 대비 비중

두 집단은 인원이 같다. 따라서 매출 비중의 차이는 고객 수가 아니라 구매 횟수와 금액에서 온다.


In [5]:
repeat_split = (analysis_customers
    .assign(구매유형=np.where(analysis_customers['다른날재방문'],'재구매 고객','구매일 1일 고객'))
    .groupby('구매유형')
    .agg(고객수=('고객ID','size'),상품매출=('Monetary','sum'))
    .reset_index())
repeat_split['상품매출'] = repeat_split['상품매출'].round(2)
repeat_split['고객 비중(%)'] = (repeat_split['고객수']/repeat_split['고객수'].sum()*100).round(2)
repeat_split['상품매출 비중(%)'] = (repeat_split['상품매출']/repeat_split['상품매출'].sum()*100).round(2)
assert repeat_split['고객수'].sum() == 1468
assert repeat_split['고객수'].tolist() == [734,734]
assert abs(repeat_split['상품매출'].sum() - 4840102.19) < 1
display(repeat_split)


,구매유형,고객수,상품매출,고객 비중(%),상품매출 비중(%)
0,구매일 1일 고객,734,1135918.47,50.0,23.47
1,재구매 고객,734,3704183.72,50.0,76.53


### 0-3. 처음 구매한 달 이후의 월별 재구매율은 어떻게 변하는가?

2019년 거래에서 첫 구매가 확인된 달을 기준으로 코호트의 경과월별 재구매율을 계산한다. 뒤쪽 경과월에는 추적 가능한 기준월이 적으므로 분모가 줄어든다.


In [6]:
history=customer_month.merge(
    analysis_customers[['고객ID','관측첫구매월']],on='고객ID',validate='many_to_one')
history['경과월']=history['구매월']-history['관측첫구매월']
cohort_data=(history.groupby(['관측첫구매월','경과월'])['고객ID']
    .nunique().reset_index(name='고객수'))
cohort_pivot=cohort_data.pivot(index='관측첫구매월',columns='경과월',values='고객수')
cohort_pct=cohort_pivot.divide(cohort_pivot[0],axis=0)*100
cohort_lag=pd.DataFrame([
    {'경과월':lag,
     '복귀고객':int(history['경과월'].eq(lag).sum()),
     '관찰가능고객':int(analysis_customers['관측첫구매월'].le(12-lag).sum())}
    for lag in range(1,12)
])
cohort_lag['복귀율(%)']=(cohort_lag['복귀고객']/cohort_lag['관찰가능고객']*100).round(2)
assert cohort_lag.loc[0,['복귀고객','관찰가능고객','복귀율(%)']].tolist()==[130,1362,9.54]
display(cohort_lag.rename(columns={'복귀고객':'재구매고객', '복귀율(%)':'재구매율(%)'}))


,경과월,재구매고객,관찰가능고객,재구매율(%)
0,1,130,1362,9.54
1,2,149,1294,11.51
2,3,142,1207,11.76
3,4,130,1129,11.51
4,5,150,994,15.09
5,6,111,900,12.33
6,7,108,763,14.15
7,8,62,651,9.52
8,9,58,488,11.89
9,10,36,311,11.58


In [7]:
hmap=cohort_pct.drop(columns=[0])
y_labels=[f'{m}월 코호트' for m in hmap.index]
x_labels=[f'+{m}개월' for m in hmap.columns]
sizes=cohort_pivot[0].reindex(hmap.index).to_numpy()
cell_text=[[f'{v:.1f}%' if pd.notna(v) and v>0 else '' for v in row]
           for row in hmap.to_numpy()]
fig=make_subplots(rows=1,cols=2,column_widths=[0.2,0.8],
    subplot_titles=('코호트 고객 수','월별 재구매율 (%)'),
    horizontal_spacing=0.04)
fig.add_trace(go.Bar(x=sizes,y=y_labels,orientation='h',text=sizes,
    textposition='auto',marker_color='rgb(150,150,150)',showlegend=False),row=1,col=1)
fig.add_trace(go.Heatmap(z=hmap.to_numpy(),x=x_labels,y=y_labels,
    colorscale='Blues',text=cell_text,texttemplate='%{text}',
    textfont=dict(size=12),colorbar=dict(title='재구매율 (%)'),zmin=0),row=1,col=2)
fig.update_yaxes(autorange='reversed')
fig.update_layout(title=dict(text='처음 구매한 달 이후의 월별 재구매율 — 전체 (2019)',
    x=0.5,xanchor='center',font=dict(size=18,color='#2c3e50')),
    plot_bgcolor='white',width=1400,height=450,margin=dict(t=140))
fig


In [8]:
fig=go.Figure(go.Scatter(
    x=cohort_lag['경과월'],y=cohort_lag['복귀율(%)'],
    mode='lines+markers',name='고객 가중 재구매율',
    line=dict(color='#3498db',width=2.5),marker=dict(size=8)))
fig.update_layout(title=dict(
    text='경과월별 재구매율: 관찰 가능 고객을 분모로 한 값',
    x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    xaxis=dict(title='2019년 첫 구매 확인월 이후 경과월',gridcolor='#ececec',
               tickvals=list(range(1,12))),
    yaxis=dict(title='재구매율 (%)',gridcolor='#ececec',range=[0,20]),
    plot_bgcolor='white',height=420)
fig


첫 구매 다음 달 재구매율은 <strong>130/1,362명(9.54%)</strong>이다. +1\~+11개월의 고객 가중 재구매율은 모두 20% 미만이지만 뒤쪽 경과월은 관찰 가능한 고객 수가 더 적다.


### 0-4. 기준선 정리

연간 재구매 고객 비율은 734/1,468명(50.0%)이고, 이 고객들이 상품매출의 76.53%를 냈다. 재구매를 관리할 이유는 분명하다.

다만 두 지표는 세는 대상이 다르다. 50.0%는 1년 안에 한 번 더 산 고객을, 첫 구매 다음 달 9.54%는 바로 다음 달에 돌아온 고객을 센다. 경과월별 재구매율은 +1\~+11개월 모두 9.52\~15.81% 범위다.

전체 평균 하나로는 1,468명 중 누구에게 먼저 접촉할지 가려지지 않는다. 다음 장부터 고객을 2019년 구매 이력으로 나눈 뒤, 등급별 가치와 복귀 패턴을 본다.


## 1. 가설 1. 고객 수가 많은 등급이 상품매출도 가장 많이 떠받칠 것이다.

<strong>분석 기준:</strong> 2019년 RFM 5등급의 고객 수 비중과 상품매출 비중을 같은 기간으로 비교하고, 이어 등급별로 고객 한 명의 누적 상품매출 분포를 본다. 분모는 전체 고객 <strong>1,468명</strong>과 전체 상품매출 <strong>$4,840,102.19</strong>(배송비 제외)다. 등급은 2019년 구매액(Monetary)을 포함해 만들었다. 이 장은 그 등급별로 2019년 상품매출이 어떻게 놓였는지를 본다.

### 1-1. 등급별 고객 수와 상품매출 비중은 같은가?


In [9]:
grade_order = ['Bronze','Silver','Gold','Platinum','Diamond']
grade = (analysis_customers.groupby('등급')
    .agg(고객수=('고객ID','size'),상품매출=('Monetary','sum'))
    .reindex(grade_order).reset_index())
grade['상품매출'] = grade['상품매출'].round(2)
grade['고객 비중(%)'] = (grade['고객수']/grade['고객수'].sum()*100).round(2)
grade['상품매출 비중(%)'] = (grade['상품매출']/grade['상품매출'].sum()*100).round(2)
assert grade.loc[0,'고객수'] == 698
assert grade.loc[0,'고객 비중(%)'] == 47.55
assert grade.loc[0,'상품매출 비중(%)'] == 10.93
top = grade[grade['등급'].isin(['Diamond','Platinum'])]
assert top['고객수'].sum() == 212
assert round(top['고객수'].sum()/grade['고객수'].sum()*100,2) == 14.44
assert round(top['상품매출'].sum()/grade['상품매출'].sum()*100,2) == 48.56
display(grade)


,등급,고객수,상품매출,고객 비중(%),상품매출 비중(%)
0,Bronze,698,529105.10,47.55,10.93
1,Silver,315,782605.60,21.46,16.17
2,Gold,243,1178150.59,16.55,24.34
3,Platinum,154,1336259.98,10.49,27.61
4,Diamond,58,1013980.92,3.95,20.95


In [10]:
color_map={'Bronze':'#CD7F32','Silver':'#C0C0C0','Gold':'#FFD700',
           'Platinum':'#E0B0FF','Diamond':'#B9F2FF'}
fig=make_subplots(rows=1,cols=2,
    subplot_titles=['등급별 고객 비중','등급별 매출 비중'],
    specs=[[{'type':'pie'},{'type':'pie'}]])
for col,value_col in enumerate(['고객 비중(%)','상품매출 비중(%)'],1):
    fig.add_trace(go.Pie(labels=grade['등급'],values=grade[value_col],
        marker=dict(colors=[color_map[g] for g in grade['등급']],
                    line=dict(color='white',width=2)),
        textinfo='label+percent',textposition='inside',
        textfont=dict(size=11),showlegend=False,hole=0.4,sort=False),row=1,col=col)
fig.update_layout(title=dict(text='등급별 고객 및 매출 비중',
    font=dict(size=22,color='#2c3e50'),x=0.5,xanchor='center'),
    height=500,plot_bgcolor='white')
fig


### 1-2. 등급별 고객 구매액 분포는 얼마나 다른가?

등급별 매출 비중에 더해 고객 한 명의 2019년 누적 상품매출 중앙값과 분포를 본다. 로그 축은 큰 지출 고객 때문에 작은 값이 눌려 보이는 문제를 줄인다.


In [11]:
grade_monetary=(analysis_customers.groupby('등급')
    .agg(고객수=('고객ID','size'),
         고객별상품매출_중앙값=('Monetary','median'),
         고객별상품매출_평균=('Monetary','mean'))
    .reindex(grade_order).reset_index())
grade_monetary[['고객별상품매출_중앙값','고객별상품매출_평균']]=(
    grade_monetary[['고객별상품매출_중앙값','고객별상품매출_평균']].round(2))
display(grade_monetary)


,등급,고객수,고객별상품매출_중앙값,고객별상품매출_평균
0,Bronze,698,629.18,758.03
1,Silver,315,2432.33,2484.46
2,Gold,243,4483.84,4848.36
3,Platinum,154,7233.87,8677.01
4,Diamond,58,12329.03,17482.43


In [12]:
fig=go.Figure()
for tier in grade_order:
    sub=analysis_customers.loc[analysis_customers['등급'].eq(tier),'Monetary']
    fig.add_trace(go.Box(y=sub,name=tier,boxpoints='outliers',
        marker=dict(color=grade_colors[tier],opacity=0.7),
        line=dict(color=grade_colors[tier]),showlegend=False))
fig.update_layout(title=dict(text='등급별 고객 누적 상품매출 분포',
    x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    xaxis_title='RFM 등급',yaxis_title='고객별 상품매출 ($, 로그 축)',
    plot_bgcolor='white',height=500)
fig.update_yaxes(type='log',gridcolor='#ececec')
fig.update_xaxes(gridcolor='#ececec')
fig


고객별 상품매출 중앙값은 Bronze 629.18달러, Silver 2,432.33달러, Gold 4,483.84달러, Platinum 7,233.87달러, Diamond 약 12,329.03달러다. 등급별 규모뿐 아니라 한 명당 누적 구매액도 다르며, 각 등급 안에도 지출 분포의 폭이 있다.


### 1-3. 가설 1에 대한 판단

고객 수가 가장 많은 Bronze 698명(47.55%)의 상품매출 비중은 10.93%였다. Platinum 154명(10.49%)이 27.61%, Diamond 58명(3.95%)이 20.95%로, 두 등급 212명(14.44%)이 48.56%를 차지했다. 고객 규모와 매출 기여의 순위는 반대였고 가설 1은 기각된다.

고객 한 명의 누적 구매액 중앙값도 Bronze 629.18달러에서 Diamond 12,329.03달러까지 벌어지며, 각 등급 안에도 분포의 폭이 있다. 등급은 2019년 구매액을 포함해 만들었다. 이 표들은 2019년 상품매출이 등급별로 어떻게 놓였는지를 보여준다.

가치 등급은 접촉 순서를 정하는 출발점이 된다. 다음은 등급마다 고객이 언제 돌아오는지다.


## 2. 가설 2. 등급이 높을수록 돌아오는 패턴도 꾸준할 것이다.

<strong>분석 기준:</strong> 등급별로 처음 구매한 달 이후의 재구매를 두 방향으로 본다. 먼저 코호트를 합쳐 경과월 단위로 묶고, 이어 등급별 코호트를 월별로 펼친다. 등급이 높으면 구매 빈도와 최근성의 수준이 높으므로 재구매율의 높낮이는 등급 산식에서 일부 따라온다. 여기서 보는 것은 수준이 아니라 경과월에 따른 곡선의 방향, 즉 각 등급이 어디서 내려가고 어디서 올라가는지다. 경과월 단위의 분모는 각 등급에서 해당 경과월을 관찰할 수 있는 고객 수이고, 코호트 단위의 분모는 그 등급·그 첫 구매월의 고객 수다.


### 2-1. 등급별 경과월 재구매율은 어떻게 변하는가?

0-3의 전체 경과월 재구매율을 등급별로 쪼갠다. 코호트는 합쳐서 경과월 단위로만 본다. 분자는 해당 경과월에 구매가 확인된 고객 수, 분모는 그 등급에서 해당 경과월을 관찰할 수 있는 고객 수다. 전체 합계는 0-3의 `cohort_lag`와 교차 검산한다.


In [13]:
grade_order = ['Bronze','Silver','Gold','Platinum','Diamond']
hist_grade = history.merge(
    analysis_customers[['고객ID','등급']],on='고객ID',validate='many_to_one')
rows=[]
for g in grade_order:
    cust_g = analysis_customers[analysis_customers['등급'].eq(g)]
    hist_g = hist_grade[hist_grade['등급'].eq(g)]
    for lag in range(1,12):
        obs = int(cust_g['관측첫구매월'].le(12-lag).sum())
        ret = int(hist_g['경과월'].eq(lag).sum())
        rows.append({'등급':g,'경과월':lag,'재구매고객':ret,'관찰가능고객':obs,
                     '재구매율(%)':round(ret/obs*100,2) if obs else None})
grade_lag = pd.DataFrame(rows)
for lag in range(1,12):
    assert grade_lag.loc[grade_lag['경과월'].eq(lag),'재구매고객'].sum() == \
           int(cohort_lag.loc[cohort_lag['경과월'].eq(lag),'복귀고객'].iloc[0])
    assert grade_lag.loc[grade_lag['경과월'].eq(lag),'관찰가능고객'].sum() == \
           int(cohort_lag.loc[cohort_lag['경과월'].eq(lag),'관찰가능고객'].iloc[0])
grade_lag_rate=(grade_lag.pivot(index='등급',columns='경과월',values='재구매율(%)')
    .reindex(grade_order))
grade_lag_obs=(grade_lag.pivot(index='등급',columns='경과월',values='관찰가능고객')
    .reindex(grade_order))
assert int(grade_lag_obs[1].sum())==1362
display(grade_lag_rate)
display(grade_lag_obs)


경과월,1,2,3,4,5,6,7,8,9,10,11
등급,,,,,,,,,,,
Bronze,4.09,5.46,5.33,4.71,4.83,3.82,2.35,1.37,1.51,0.00,2.06
Silver,11.93,11.70,10.25,12.83,15.87,11.73,16.00,8.20,6.59,3.70,8.33
Gold,13.43,17.91,17.28,15.22,19.30,13.64,20.59,9.09,10.00,11.27,23.26
Platinum,17.93,20.45,24.17,25.86,33.33,32.69,30.11,26.92,28.57,34.88,44.00
Diamond,25.00,36.36,44.23,33.33,57.45,43.18,46.51,41.03,60.00,47.83,57.14


경과월,1,2,3,4,5,6,7,8,9,10,11
등급,,,,,,,,,,,
Bronze,660,641,600,552,476,419,341,291,199,120,97
Silver,285,265,244,226,189,179,150,122,91,54,36
Gold,216,201,191,184,171,154,136,121,100,71,43
Platinum,145,132,120,116,111,104,93,78,63,43,25
Diamond,56,55,52,51,47,44,43,39,35,23,14


In [14]:
MIN_OBS=30
dropped={g:grade_lag.loc[grade_lag['등급'].eq(g)
    &grade_lag['관찰가능고객'].lt(MIN_OBS),'경과월'].tolist() for g in grade_order}
assert dropped=={'Bronze':[],'Silver':[],'Gold':[],'Platinum':[11],'Diamond':[10,11]}
fig=go.Figure()
fig.add_trace(go.Scatter(x=cohort_lag['경과월'],y=cohort_lag['복귀율(%)'],
    mode='lines',name='전체 평균',line=dict(color='#95a5a6',width=2,dash='dash'),
    hovertemplate='전체 · +%{x}개월<br>재구매율 %{y:.2f}%<extra></extra>'))
for g in grade_order:
    sub=grade_lag[grade_lag['등급'].eq(g)&grade_lag['관찰가능고객'].ge(MIN_OBS)]
    fig.add_trace(go.Scatter(x=sub['경과월'],y=sub['재구매율(%)'],
        mode='lines+markers',name=g,
        line=dict(color=grade_colors[g],width=2.5),
        marker=dict(size=7,line=dict(color='#2c3e50',width=0.6)),
        customdata=np.stack([sub['재구매고객'],sub['관찰가능고객']],axis=-1),
        hovertemplate=(g+' · +%{x}개월<br>재구매율 %{y:.2f}%'
            '<br>재구매 %{customdata[0]}명 / 관찰가능 %{customdata[1]}명<extra></extra>')))
fig.update_layout(title=dict(
    text='등급별 경과월 재구매율 — 관찰 가능 고객을 분모로 한 값'
         '<br><sub>회색 점선 = 전체 평균 · 관찰가능고객 30명 미만 구간 제외</sub>',
    x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    xaxis=dict(title='2019년 첫 구매 확인월 이후 경과월',gridcolor='#ececec',
               tickvals=list(range(1,12))),
    yaxis=dict(title='재구매율 (%)',gridcolor='#ececec',rangemode='tozero'),
    plot_bgcolor='white',height=520,margin=dict(t=120),
    legend=dict(orientation='v',x=1.01,y=1))
fig


등급별 +1개월 재구매율은 Bronze 4.09%, Silver 11.93%, Gold 13.43%, Platinum 17.93%, Diamond 25.00%다. Bronze는 +2개월 5.46%가 가장 높고 +6개월 3.82%부터 내려가 +10개월 0.00%가 된다. Silver와 Gold는 +7개월까지 각각 10.25\~16.00%, 13.43\~20.59% 사이를 오르내리다 +8개월 8.20%·9.09%로 내려간다. Platinum과 Diamond는 내려가는 구간 없이 각각 +11개월 44.00%, +9개월 60.00%까지 높은 수준을 유지한다.

차트에서 제외한 구간은 관찰가능고객 30명 미만인 Platinum +11개월(25명), Diamond +10개월(23명)·+11개월(14명)이다. 표에는 전 구간을 남겼다. 뒤쪽 경과월은 모든 등급에서 분모가 줄어든다. +11개월의 관찰가능고객은 Bronze 97명, Silver 36명, Gold 43명이다.


### 2-2. 등급별 코호트 리텐션은 어떻게 다른가?

2-1은 코호트를 합쳐 경과월 단위로 봤다. 여기서는 등급별로 코호트를 첫 구매월까지 펼친다. 행은 2019년 거래에서 첫 구매가 확인된 달, 열은 경과월, 값은 그 코호트에서 해당 경과월에 재구매한 고객 비율이다. 분모는 교차표의 코호트 크기다.


In [15]:
grade_cohort = (analysis_customers
    .pivot_table(index='등급',columns='관측첫구매월',values='고객ID',
                 aggfunc='size',fill_value=0)
    .reindex(grade_order))
assert grade_cohort.values.sum() == 1468
assert grade_cohort.sum(axis=1).tolist() == [698,315,243,154,58]
display(grade_cohort)


관측첫구매월,1,2,3,4,5,6,7,8,9,10,11,12
등급,,,,,,,,,,,,
Bronze,97,23,79,92,50,78,57,76,48,41,19,38
Silver,36,18,37,31,28,29,10,37,18,21,20,30
Gold,43,28,29,21,15,18,17,13,7,10,15,27
Platinum,25,18,20,15,15,11,7,5,4,12,13,9
Diamond,14,9,12,4,4,1,3,4,1,3,1,2


In [16]:
import plotly.colors as pc
months=list(range(1,13)); lags=list(range(1,12))
observable=pd.DataFrame({l:[m+l<=12 for m in months] for l in lags},index=months)
grade_cohort_pct={}; grade_cohort_n={}; zero_cells={}
for g in grade_order:
    h=hist_grade[hist_grade['등급'].eq(g)]
    n=(h.groupby(['관측첫구매월','경과월'])['고객ID'].nunique()
        .reset_index(name='고객수')
        .pivot(index='관측첫구매월',columns='경과월',values='고객수')
        .reindex(index=months,columns=lags)
        .fillna(0).where(observable))
    size=grade_cohort.loc[g].reindex(months)
    grade_cohort_n[g]=n
    grade_cohort_pct[g]=n.divide(size,axis=0)*100
    zero_cells[g]=int(n.eq(0).sum().sum())
assert all(int(grade_cohort_pct[g].notna().sum().sum())==66 for g in grade_order)
assert zero_cells=={'Bronze':14,'Silver':5,'Gold':4,'Platinum':2,'Diamond':15}
ZMAX=float(np.nanmax([grade_cohort_pct[g].to_numpy(dtype=float) for g in grade_order]))
GAMMA=0.75
_pos=np.linspace(0,1,64)
CSCALE=[[float(t),col] for t,col in
        zip(_pos,pc.sample_colorscale('Blues',list(_pos**GAMMA)))]
fig=make_subplots(rows=5,cols=1,shared_xaxes=True,vertical_spacing=0.035,
    subplot_titles=[f'{g} · 고객 {int(grade_cohort.loc[g].sum())}명'
                    for g in grade_order])
for r,g in enumerate(grade_order,1):
    pct=grade_cohort_pct[g].to_numpy(dtype=float)
    cnt=grade_cohort_n[g].to_numpy(dtype=float)
    size=grade_cohort.loc[g].reindex(months).to_numpy(dtype=float)
    cell_text=[[f'{v:.0f}% ({c:.0f}명)' if pd.notna(v) else ''
                for v,c in zip(prow,crow)] for prow,crow in zip(pct,cnt)]
    fig.add_trace(go.Heatmap(z=pct,
        x=[f'+{l}개월' for l in lags],
        y=[f'{m}월 · {int(s)}명' for m,s in zip(months,size)],
        colorscale=CSCALE,zmin=0,zmax=ZMAX,xgap=1,ygap=1,
        text=cell_text,texttemplate='%{text}',textfont=dict(size=9),
        customdata=np.dstack([np.tile(size[:,None],(1,len(lags))),cnt]),
        hovertemplate=(g+' · %{y} 코호트 · %{x}<br>재구매율 %{z:.1f}%'
            '<br>재구매 %{customdata[1]:.0f}명 / 코호트 %{customdata[0]:.0f}명'
            '<extra></extra>'),
        showscale=(r==1),colorbar=dict(title='재구매율 (%)',len=0.4,y=0.8)),
        row=r,col=1)
fig.update_yaxes(autorange='reversed',title_text='첫 구매월 · 코호트 크기')
fig.update_layout(title=dict(
    text='등급별 코호트 리텐션 — 첫 구매월 × 경과월'
         '<br><sub>색 기준 공유(0~%.1f%%, 낮은 구간 대비 강조) · 복귀 고객이 없는 칸은 0%% · '
         '회색 칸은 2019년 안에 관찰할 수 없는 구간 · 행 라벨의 명수가 분모</sub>'%ZMAX,
    x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    plot_bgcolor='#ececec',paper_bgcolor='white',
    width=1150,height=1750,margin=dict(t=150))
fig


코호트 크기는 등급에 따라 크게 다르다. Bronze는 월 19\~97명, Silver 10\~37명, Gold 7\~43명, Platinum 4\~25명, Diamond 1\~14명이다. 관찰 가능한 칸은 등급마다 66개이고 모두 값을 적었다. 분모가 되는 코호트 크기는 행 라벨에 함께 썼다. 복귀 고객이 한 명도 없는 칸은 0%로 적었고 Bronze 14칸, Silver 5칸, Gold 4칸, Platinum 2칸, Diamond 15칸이다. 회색 칸은 첫 구매월과 경과월을 더하면 12를 넘어 2019년 안에 관찰되지 않는 구간이다.

코호트가 작은 칸은 비율이 한두 명으로 움직인다. Diamond 6월 코호트의 +1개월 100%는 1명 중 1명이고, 9월과 11월의 0%도 1명 중 0명이다. Platinum 7\~9월과 12월 코호트도 4\~9명이다. 등급별 비교는 코호트 10명 이상인 행, 즉 Bronze 전체와 Silver 전체, Gold 9월을 뺀 행, Platinum 1\~6월과 10\~11월, Diamond 1월과 3월을 중심으로 읽는다. 2-1에서 코호트를 합쳐 경과월 단위로 본 것도 같은 이유다.

교차표에는 등급별 코호트 쏠림이 있다. Diamond 58명 중 1\~3월 코호트가 35명(60.3%)이고 Platinum은 154명 중 63명(40.9%), Gold는 243명 중 100명(41.2%)이다. Bronze는 698명 중 199명(28.5%), Silver는 315명 중 91명(28.9%)으로 더 고르게 퍼져 있다. 등급은 2019년 전체의 누적 Frequency·Monetary로 매기므로 연초 코호트일수록 누적 기간이 길다. 상위 등급이 연초 코호트에 몰린 방향은 이 누적 기간 차이와 같은 방향이다.


### 2-3. 가설 2에 대한 판단

등급별 +1개월 재구매율은 Bronze 4.09%, Silver 11.93%, Gold 13.43%, Platinum 17.93%, Diamond 25.00%로 전체 평균 9.54%를 사이에 두고 갈린다. Bronze만 평균 아래에 있다.

수준뿐 아니라 곡선의 모양이 다르다. Bronze는 앞쪽 경과월에서 4\~5%대로 평평하게 유지되다 경과월 7부터 내려가고, 상위 등급은 뒤쪽으로 갈수록 올라간다. Bronze는 경과월 7부터 2.35%·1.37%·1.51%·0.00%·2.06%로 내려간다. Platinum은 17.93%에서 44.00%로, Diamond는 25.00%에서 57.14%로 올라간다. Silver와 Gold는 10\~20% 구간에서 유지되다 경과월 8 부근에서 8.20%·9.09%로 내려간다.

등급 산식에는 Frequency와 Recency의 수준이 들어가므로 재구매율의 높낮이는 일부 따라온다. 여기서 확인한 것은 경과월에 따른 곡선의 방향이고, 그것은 점수에 들어있지 않다. 가설 2는 지지된다.

뒤쪽 경과월은 분모가 줄어드는 구간이면서, 2-2에서 본 등급별 코호트 쏠림과 연말이 함께 겹친다. 상위 등급의 뒤쪽 칸은 연초 코호트가 대부분이고 Platinum +11개월 25명, Diamond +11개월 14명은 2-1 차트에서 빠졌다. 따라서 판단의 주 근거는 분모가 큰 앞쪽 경과월 1\~6에 둔다. 그 구간에서도 Bronze 4.09\~5.46%, Platinum 17.93\~33.33%, Diamond 25.00\~57.45%로 방향이 갈린다.

접촉 시점 후보는 두 갈래로 나온다. Bronze는 +1\~+2개월에 재구매가 가장 높고 +7개월부터 2% 아래로 내려가므로, 첫 구매 직후 몇 달에 접촉을 모으는 4장 P1으로 이어진다. 상위 등급은 공백이 길어도 돌아오므로, 시점을 앞당기는 대신 메시지와 혜택 방식을 비교하는 4장 P3·P4로 이어진다.


## 3. 가설 3. 같은 등급 안에서는 구매 상태가 비슷할 것이다.

<strong>분석 기준:</strong> 각 등급의 행동 세그먼트 구성을 보고, 등급 안의 집단을 고객 수와 Recency(일), Frequency(고객 주문 건수), Monetary(달러) 원지표의 중앙값으로 비교한다. 등급은 가치 수준을, 세그먼트는 2019년 R·F·M 점수 조합에 따른 기준일의 구매 상태를 가리킨다. 집단 사이의 Recency 차이는 R 점수로 나눈 결과이므로, 확인하는 것은 그 안에서 Monetary가 함께 낮아지는지다.

### 3-1. 등급별 행동 세그먼트 구성은 어떤가?


In [17]:
grade_order=['Bronze','Silver','Gold','Platinum','Diamond']
segment_counts=(analysis_customers.groupby(['등급','세그먼트']).size()
    .rename('고객수').reset_index())
segment_mix=(segment_counts.pivot(index='등급',columns='세그먼트',values='고객수')
    .reindex(grade_order).fillna(0).astype(int))
segment_pct=segment_mix.div(segment_mix.sum(axis=1),axis=0)*100
assert segment_mix.sum().sum()==1468
assert segment_mix.loc['Diamond','VIP 고객']==58
assert segment_mix.loc['Platinum',['VIP 고객','충성 고객']].sum()==147


In [18]:
segment_colors={
    'VIP 고객':'#1a9850','충성 고객':'#66bd63','잠재 충성 고객':'#a6d96a',
    '최근·저빈도 고객':'#3498db','가망 고객':'#74b9ff','관심 필요 고객':'#fdae61',
    '이탈 조짐 고객':'#f4a261','놓치면 안될 고객':'#9b59b6',
    '이탈 위험 고객':'#e74c3c','휴면 고객':'#95a5a6','이탈 고객':'#636e72'}
fig=go.Figure()
for seg in segment_mix.sum().sort_values(ascending=False).index:
    pct=segment_pct[seg].to_numpy()
    cnt=segment_mix[seg].to_numpy()
    fig.add_trace(go.Bar(x=grade_order,y=pct,name=seg,
        marker_color=segment_colors[seg],
        text=[f'{v:.1f}%' if v>=8 else '' for v in pct],
        textposition='inside',customdata=cnt,
        hovertemplate=f'{seg}<br>%{{y:.1f}}% · %{{customdata}}명<extra></extra>'))
fig.update_layout(barmode='stack',
    title=dict(text='등급 내 행동 세그먼트 구성',
        x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    xaxis_title='RFM 등급',yaxis_title='등급 내 고객 비율 (%)',
    plot_bgcolor='white',height=560,legend=dict(orientation='v',x=1.01,y=1))
fig.update_yaxes(range=[0,100],gridcolor='#ececec')
fig


Diamond 58명은 모두 VIP 고객이다. Platinum은 VIP 77명과 충성 70명이 147/154명(95.5%)을 차지한다. Bronze는 휴면 295명, 최근·저빈도 220명, 이탈 129명이 대부분이다. Silver와 Gold는 여러 행동 상태가 함께 있다. 각 등급 안의 구매 공백과 누적 구매액은 3-2부터 차례로 본다.


### 3-2. Bronze 안에서는 어떤 구매 상태가 나뉘는가?

고객 수가 가장 많은 Bronze의 주요 세그먼트를 고객 수, 마지막 구매 공백, 누적 상품매출로 비교한다. 가망 4명과 놓치면 안될 3명은 표에 함께 싣고, 해석은 인원이 큰 상위 세 세그먼트를 중심으로 한다.


In [19]:
bronze_order=['최근·저빈도 고객','휴면 고객','이탈 고객','이탈 조짐 고객','가망 고객','놓치면 안될 고객']
bronze_profile=(analysis_customers[analysis_customers['등급'].eq('Bronze')]
    .groupby('세그먼트')
    .agg(고객수=('고객ID','size'),Recency_중앙값_일=('Recency','median'),
         Frequency_중앙값_건=('Frequency','median'),Monetary_중앙값_달러=('Monetary','median'),
         다른날재방문고객수=('다른날재방문','sum'))
    .reindex(bronze_order).reset_index())
bronze_profile['Monetary_중앙값_달러']=bronze_profile['Monetary_중앙값_달러'].round(2)
bronze_profile['다른날재방문율(%)']=(bronze_profile['다른날재방문고객수']/bronze_profile['고객수']*100).round(1)
display(bronze_profile.rename(columns={'다른날재방문고객수':'재구매 고객 수', '다른날재방문율(%)':'재구매 고객 비율(%)'}))

,세그먼트,고객수,Recency_중앙값_일,Frequency_중앙값_건,Monetary_중앙값_달러,재구매 고객 수,재구매 고객 비율(%)
0,최근·저빈도 고객,220,81.0,3.0,440.35,65,29.5
1,휴면 고객,295,202.0,7.0,1023.75,89,30.2
2,이탈 고객,129,290.0,3.0,346.67,8,6.2
3,이탈 조짐 고객,47,144.0,7.0,579.40,20,42.6
4,가망 고객,4,22.0,6.5,592.68,2,50.0
5,놓치면 안될 고객,3,268.0,4.0,2097.92,0,0.0


Bronze의 최근·저빈도 고객 220명은 Recency 중앙값 81일, Monetary 440.35달러다. 휴면 고객 295명은 202일·1,023.75달러, 이탈 고객 129명은 290일·346.67달러다. 재구매 고객 비율(서로 다른 날 2일 이상 구매)도 이탈 고객 6.2%, 이탈 조짐 고객 42.6%로 다르다. 같은 Bronze라도 최근에 적게 구매한 고객, 누적 구매가 있으나 공백이 긴 고객, 장기간 구매가 뜸한 고객이 섞여 있다. '휴면'과 '이탈'은 2019년 R·F·M 점수 조합으로 붙인 세그먼트 이름이며, 기준일 구매 공백의 길이를 가리킨다.

### 3-3. Silver·Gold 안에서 구매 공백과 누적 구매액은 어떻게 다른가?

- 집단 1: Silver·Gold의 잠재 충성 고객
- 집단 2: 같은 등급의 이탈 위험 고객
- 기준: Recency(일) · Frequency(건) · Monetary(달러) 원지표

산점도의 마커 크기는 Frequency 중앙값이다.


In [20]:
within = (analysis_customers[
    analysis_customers['등급'].isin(['Silver','Gold']) &
    analysis_customers['대표상태'].isin(['잠재 충성','이탈 위험'])
].groupby(['등급','대표상태'])
 .agg(고객수=('고객ID','size'),Recency_중앙값=('Recency','median'),
      Frequency_중앙값=('Frequency','median'),Monetary_중앙값=('Monetary','median'))
 .reindex(pd.MultiIndex.from_product([['Silver','Gold'],['잠재 충성','이탈 위험']],
                                      names=['등급','대표상태'])).reset_index())
from decimal import Decimal, ROUND_HALF_EVEN
within['Monetary_중앙값'] = within['Monetary_중앙값'].map(
    lambda x: float(Decimal(str(x)).quantize(Decimal('0.01'), rounding=ROUND_HALF_EVEN))
)
expected = [
    ('Silver','잠재 충성',140,91.5,16,2482.12),
    ('Silver','이탈 위험',55,279.0,19,3271.17),
    ('Gold','잠재 충성',65,20.0,15,2789.52),
    ('Gold','이탈 위험',44,190.5,32,6213.04),
]
for row, truth in zip(within.itertuples(index=False,name=None),expected):
    assert row[:5] == truth[:5]
    assert round(row[5],2) == truth[5]
display(within)


,등급,대표상태,고객수,Recency_중앙값,Frequency_중앙값,Monetary_중앙값
0,Silver,잠재 충성,140,91.5,16.0,2482.12
1,Silver,이탈 위험,55,279.0,19.0,3271.17
2,Gold,잠재 충성,65,20.0,15.0,2789.52
3,Gold,이탈 위험,44,190.5,32.0,6213.04


In [21]:
bubble=within.copy()
fig=make_subplots(rows=1,cols=2,
    subplot_titles=['Silver: Recency vs Monetary','Gold: Recency vs Monetary'])
palette={
    ('Silver','잠재 충성'):'#2ecc71',('Silver','이탈 위험'):'#e67e22',
    ('Gold','잠재 충성'):'#a6d96a',('Gold','이탈 위험'):'#e74c3c'
}
for col,tier in enumerate(['Silver','Gold'],1):
    for _,row in bubble[bubble['등급']==tier].iterrows():
        fig.add_trace(go.Scatter(
            x=[row['Recency_중앙값']],y=[row['Monetary_중앙값']],
            mode='markers+text',text=[row['대표상태']],textposition='top center',
            marker=dict(size=row['Frequency_중앙값']*3+10,
                color=palette[(tier,row['대표상태'])],
                line=dict(color='#2c3e50',width=0.8),opacity=0.85),
            showlegend=False,name=row['대표상태']
        ),row=1,col=col)
fig.update_xaxes(title_text='Recency 중앙값 (일)',gridcolor='#ececec')
fig.update_yaxes(title_text='Monetary 중앙값 ($)',gridcolor='#ececec')
fig.update_layout(title=dict(
    text='등급 내부 구매 공백과 누적 상품매출<br><sub>마커 크기 = Frequency 중앙값</sub>',
    font=dict(size=18,color='#2c3e50'),x=0.5,xanchor='center'),
    plot_bgcolor='white',width=1050,height=520)
fig


In [22]:
fig=make_subplots(rows=2,cols=2,
    subplot_titles=['Silver: 마지막 구매 공백','Gold: 마지막 구매 공백',
                    'Silver: 누적 상품매출','Gold: 누적 상품매출'],
    vertical_spacing=0.2)
for col,tier in enumerate(['Silver','Gold'],1):
    for state,color in [('잠재 충성','#2ecc71'),('이탈 위험','#e74c3c')]:
        sub=analysis_customers.loc[
            analysis_customers['등급'].eq(tier)&analysis_customers['대표상태'].eq(state)]
        fig.add_trace(go.Box(y=sub['Recency'],name=state,boxpoints='outliers',
            marker=dict(color=color),line=dict(color=color),
            showlegend=(col==1),legendgroup=state),row=1,col=col)
        fig.add_trace(go.Box(y=sub['Monetary'],name=state,boxpoints='outliers',
            marker=dict(color=color),line=dict(color=color),
            showlegend=False,legendgroup=state),row=2,col=col)
fig.update_layout(title=dict(text='같은 등급 내부의 구매 상태 분포',
    x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    plot_bgcolor='white',height=680,boxmode='group')
fig.update_yaxes(title_text='Recency (일)',gridcolor='#ececec',row=1,col=1)
fig.update_yaxes(gridcolor='#ececec',row=1,col=2)
fig.update_yaxes(title_text='Monetary ($, 로그 축)',type='log',gridcolor='#ececec',row=2,col=1)
fig.update_yaxes(type='log',gridcolor='#ececec',row=2,col=2)
fig


### 3-4. Platinum과 Diamond 안에서도 다른 상태가 보이는가?

Platinum의 VIP·충성 고객과 구매 공백이 특히 긴 고객을 나눠 보고, Diamond는 단일 VIP 세그먼트 안의 분포를 확인한다.


In [23]:
upper_order=pd.MultiIndex.from_tuples([
    ('Platinum','VIP 고객'),('Platinum','충성 고객'),
    ('Platinum','이탈 위험 고객'),('Platinum','놓치면 안될 고객'),
    ('Diamond','VIP 고객')],names=['등급','세그먼트'])
upper_profile=(analysis_customers[analysis_customers['등급'].isin(['Platinum','Diamond'])]
    .groupby(['등급','세그먼트'])
    .agg(고객수=('고객ID','size'),Recency_중앙값_일=('Recency','median'),
         Frequency_중앙값_건=('Frequency','median'),Monetary_중앙값_달러=('Monetary','median'),
         구매공백_90일초과_고객수=('장기미방문_90일예시','sum'))
    .reindex(upper_order).reset_index())
upper_profile['Monetary_중앙값_달러']=upper_profile['Monetary_중앙값_달러'].round(2)
display(upper_profile)

,등급,세그먼트,고객수,Recency_중앙값_일,Frequency_중앙값_건,Monetary_중앙값_달러,구매공백_90일초과_고객수
0,Platinum,VIP 고객,77,26.0,36.0,7681.24,1
1,Platinum,충성 고객,70,78.5,36.5,6208.44,25
2,Platinum,이탈 위험 고객,4,185.5,62.5,10759.84,4
3,Platinum,놓치면 안될 고객,3,272.0,65.0,15042.41,3
4,Diamond,VIP 고객,58,31.5,71.0,12329.03,2


Platinum은 VIP 77명과 충성 70명이 147/154명(95.5%)이다. 기준일 구매 공백이 90일을 넘는 고객은 전체 33명이며, 이 중 이탈 위험 4명과 놓치면 안될 3명은 Recency 중앙값이 각각 185.5일과 272일이다. 두 소수 집단의 Monetary 중앙값은 각각 10,759.84달러와 15,042.41달러로, 구매액이 높은 고객의 긴 공백을 별도로 살펴볼 이유가 있다. 두 집단은 각각 4명과 3명이며, 위 수치는 그 인원의 중앙값이다.

Diamond 58명은 모두 VIP 고객이다. Recency 중앙값은 31.5일이고 90일 초과 고객은 2명이다. Monetary의 중앙 50%는 약 10,991\~17,477달러에 분포한다. Diamond는 등급·세그먼트·Recency가 한 덩어리로 묶인다.

### 3-5. Platinum의 현재 구매 공백과 복귀 이력은 다른가?

- 집단 1: 기준일 Recency가 90일을 넘는 Platinum 고객
- 집단 2: 연중 서로 다른 구매일 사이의 최대 간격이 90일을 넘고 다시 구매한 고객
- 집단 3: 그 외 Platinum 고객
- 기준: Recency(일) · 구매일수 · Monetary(달러) 원지표

90일은 공백의 길이를 가르는 관찰 기준으로 쓴다. 같은 고객에게 두 조건이 모두 해당하면 기준일 공백을 먼저 표시한다.


In [24]:
purchase_days=(analysis_orders.assign(구매일=analysis_orders['거래날짜'].dt.normalize())
    [['고객ID','구매일']].drop_duplicates().sort_values(['고객ID','구매일']))
purchase_days['구매간격_일']=purchase_days.groupby('고객ID')['구매일'].diff().dt.days
max_purchase_gap=(purchase_days.groupby('고객ID')['구매간격_일'].max()
    .fillna(0).rename('최대구매간격_일'))
platinum_gap=(analysis_customers[analysis_customers['등급'].eq('Platinum')]
    .join(max_purchase_gap,on='고객ID'))
platinum_gap['공백상태']='그 외 Platinum'
platinum_gap.loc[platinum_gap['최대구매간격_일'].gt(90),'공백상태']='장기 공백 후 복귀'
platinum_gap.loc[platinum_gap['Recency'].gt(90),'공백상태']='기준일 90일 초과'
gap_state_order=['그 외 Platinum','장기 공백 후 복귀','기준일 90일 초과']
platinum_gap_summary=(platinum_gap.groupby('공백상태')
    .agg(고객수=('고객ID','size'),Recency_중앙값_일=('Recency','median'),
         구매일수_중앙값=('구매일수','median'),Monetary_중앙값_달러=('Monetary','median'))
    .reindex(gap_state_order).reset_index())
platinum_gap_summary['Monetary_중앙값_달러']=platinum_gap_summary['Monetary_중앙값_달러'].round(2)
assert platinum_gap_summary['고객수'].tolist()==[49,72,33]
assert int((platinum_gap['최대구매간격_일'].gt(90)|platinum_gap['Recency'].gt(90)).sum())==105
display(platinum_gap_summary)

,공백상태,고객수,Recency_중앙값_일,구매일수_중앙값,Monetary_중앙값_달러
0,그 외 Platinum,49,51.0,2.0,6023.50
1,장기 공백 후 복귀,72,34.0,4.0,7493.67
2,기준일 90일 초과,33,121.0,4.0,10653.02


In [25]:
fig=make_subplots(rows=1,cols=3,subplot_titles=['Recency (일)','구매일수','누적 상품매출 (달러)'])
state_colors={'그 외 Platinum':'#E0B0FF','장기 공백 후 복귀':'#3498db','기준일 90일 초과':'#e74c3c'}
for col,metric in enumerate(['Recency','구매일수','Monetary'],1):
    for state in gap_state_order:
        sub=platinum_gap.loc[platinum_gap['공백상태'].eq(state),metric]
        fig.add_trace(go.Box(y=sub,name=state,boxpoints='outliers',
            marker_color=state_colors[state],line_color=state_colors[state],
            showlegend=(col==1),legendgroup=state),row=1,col=col)
fig.update_layout(title=dict(text='Platinum 고객 유형별 구매 패턴 비교',
    x=0.5,xanchor='center',font=dict(size=20,color='#2c3e50')),
    plot_bgcolor='white',height=520,width=1050,boxmode='group',
    legend=dict(orientation='v',x=1.01,y=1))
fig.update_yaxes(gridcolor='#ececec')
fig.update_yaxes(type='log',row=1,col=3)
fig


Platinum 154명 중 기준일 구매 공백이 90일을 넘는 고객은 33명, 연중 90일 넘는 구매 간격을 겪고 다시 구매한 고객은 72명, 나머지는 49명이다. 현재 공백과 과거 공백을 합치면 105/154명(68.2%)이다. 누적 상품매출 중앙값은 순서대로 10,653.02달러, 7,493.67달러, 6,023.50달러다. 누적 구매액이 가장 큰 집단이 현재 공백도 가장 길다. 이 구분은 기준일의 Recency와 연중 최대 구매 간격을 각각 90일로 가른 결과다.

### 3-6. 가설 3에 대한 판단

같은 등급 안에서 구매 상태가 갈렸고 가설 3은 기각된다.

Bronze 698명은 최근·저빈도 220명(공백 81일·440.35달러), 휴면 295명(202일·1,023.75달러), 이탈 129명(290일·346.67달러)으로 나뉘고 재구매 고객 비율도 6.2%에서 42.6%까지 다르다.

Silver의 이탈 위험 55명은 공백 279일·3,271.17달러로, 잠재 충성 140명의 91.5일·2,482.12달러보다 누적 구매액이 더 컸다. Gold도 이탈 위험 44명이 190.5일·6,213.04달러로 잠재 충성 65명의 20일·2,789.52달러를 넘었다. Platinum에서도 기준일 공백 90일 초과 33명의 누적 상품매출 중앙값 10,653.02달러가 장기 공백 후 복귀 72명의 7,493.67달러, 그 외 49명의 6,023.50달러보다 높았다.

세 등급에서 같은 방향의 역전이 나타났다. 구매 공백이 긴 쪽의 누적 구매액이 더 크다. 최근 구매일만으로 대상을 추리면 과거 구매액이 큰 고객이 빠진다. Diamond 58명은 모두 VIP 세그먼트이고 Recency 중앙값 31.5일로 내부 구분이 나타나지 않았다.

접촉 대상은 등급 하나나 구매 공백 하나가 아니라 두 축의 조합으로 뽑는다.


## 4. 누구에게 언제 접촉하는가

1장에서 가치 등급이 접촉 순서의 출발점임을, 2장에서 등급별 복귀 패턴을, 3장에서 등급 내부의 구매 상태 차이를 확인했다. 세 결과를 합쳐 등급과 현재 상태의 조합으로 다음 네 가지 비교를 제안한다. 아래 인원은 2019년 관찰 사례다.

- <strong>P1 · 관측 초기·Bronze 재참여:</strong> 최근·저빈도 220명을 초기 재구매 후보로, 휴면 295명을 별도 재참여 검토군으로 본다. 2019년 거래에서 첫 구매가 확인된 뒤 <strong>30일 vs 60일 메시지</strong>를 비교한다. 후속 30일의 다른 날 재구매율을 확인한다. <em>2-1에서 Bronze는 +1개월 4.09%, +2개월 5.46%로 초기 두 달이 가장 높고 +7개월부터 2.35% 아래로 내려간다. 30일·60일은 이 구간 안의 시점이다.</em>
- <strong>P2 · Silver·Gold 휴면 방지:</strong> 이탈 위험 Silver 55명·Gold 44명을 참고해, 마지막 구매 후 <strong>60일 vs 90일 vs 120일 접촉</strong>을 비교한다. 후속 30일 복귀율·마진·수신 거부율을 확인한다. <em>2-1 곡선은 첫 구매 기준 경과월이고 이 접촉 시점은 마지막 구매 기준 일수여서 시계가 다르다. Silver·Gold가 경과월 8 부근에서 내려간다는 점만 참고하고, 60·90·120일은 실험 후보로 둔다.</em>
- <strong>P3 · Platinum 메시지 개인화:</strong> 현재 구매 공백이 90일을 넘는 33명을 참고해 <strong>일반 메시지 vs 구매 이력 기반 메시지</strong>를 비교한다. 후속 30일 복귀율·할인 비용률을 확인한다. <em>2-1에서 Platinum은 +1개월 17.93%에서 뒤쪽으로 갈수록 올라가 공백이 길어도 돌아온다. 접촉 시점을 앞당기는 대신 메시지 방식을 먼저 비교할 근거가 된다.</em>
- <strong>P4 · Diamond·Platinum 혜택 방식:</strong> 두 등급 212명을 대상으로 <strong>쿠폰 vs 등급 혜택</strong>을 비교한다. 두 군 간 증분 재구매율·객단가를 확인한다. <em>2-1에서 Diamond는 +3개월 이후 33.33\~60.00%를 유지한다. 혜택 방식 비교는 접촉 시점 조건 없이 설계한다.</em>

실행 전에는 실제 첫 구매 여부, 접촉 가능 고객 수, 각 군의 표본 크기와 동일한 추적 기간을 확인한다. 특히 Platinum 33명은 작아 모집 범위를 넓혀야 할 수 있다. 네 항목은 모두 비교 설계이며, 어느 방식이 더 효과적인지는 그 비교의 결과로 정한다.


## 5. 분석 범위와 활용 방향

관측 기간은 2019년 한 해이고 기준일은 2019-12-31이다. 이 범위에서 따라오는 해석의 한계를 모아 둔다.

- 첫 구매는 2019년 거래에서 처음 확인된 구매다. 실제 생애 첫 구매는 이 자료로 확인할 수 없다.
- RFM 등급과 11개 행동 세그먼트는 2019-12-31 시점의 구매 이력 프로파일이며, 미래 고객 가치나 이탈을 예측하는 모델은 아니다. 등급 산식에 Recency·Frequency·Monetary가 모두 들어가므로 등급별 매출 비중과 재구매율의 수준 차이는 일부가 산식에서 따라온다.
- 경과월별 재구매율은 뒤쪽 경과월에서 분모가 줄어든다. 2-1에서 관찰가능고객 30명 미만인 Platinum +11개월(25명), Diamond +10개월(23명)·+11개월(14명)은 차트에서 제외했다.
- 2-2의 등급별 코호트는 상위 등급에서 월 한 자릿수까지 작아진다. Diamond 6월처럼 코호트가 1명인 칸의 비율은 그 한 명의 행동이고, 분모는 행 라벨에 적었다. 등급을 2019년 전체 누적 F·M으로 매기므로 상위 등급은 누적 기간이 긴 연초 코호트에 몰린다.
- Platinum의 이탈 위험 4명, 놓치면 안될 3명처럼 인원이 한 자릿수인 집단의 중앙값은 그 인원 위에서만 읽는다.
- 캠페인 발송과 노출, 그 이후의 구매, 마진 자료가 없다. 초기 재구매의 인과 요인과 실제 접촉 효과는 이 자료 밖에 있다. Recency 90일 초과는 공백의 길이를 가르는 관찰 기준이며 검증된 이탈 임계값은 아니다.

후속 분석에서는 실제 첫 구매와 추적 기간을 확인하고, 과거 기간으로 만든 등급을 이후 행동에 대조하며, 캠페인 효과는 무작위 배정 후 같은 관찰창에서 측정한다.
